# CV Masterclass 10: Production Augmentations

Welcome to Notebook 10. The pristine academic datasets (ImageNet, COCO) are a lie. They are perfectly lit, high-resolution `.png` files captured by professional cameras.

Production data comes from $10 webcam sensors covered in dust, mounted on vibrating factory machinery, streaming aggressively compressed MJPEG video over weak Wi-Fi.

If you do not simulate this violence during training, your $99\%$ accurate CNN will instantly drop to $30\%$ accuracy in production.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin:

> *"Mathematically, what destroys a CNN's feature maps when a clear PNG is converted to an aggressively compressed MJPEG video stream?"*

---

## Table of Contents
1. **The MJPEG Assassin:** Discrete Cosine Transforms & Artifacts.
2. **Albumentations:** Building a brutal physical pipeline.
3. **MixUp & CutMix:** Destroying the spatial prior to force generalization.


## 1. The MJPEG Assassin

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why does MJPEG compression destroy CNNs?*

**A:** 
Deep Neural Networks are essentially mathematical edge detectors. 
JPEG compression does not uniformly lower the resolution of an image. It slices the image into $8\times8$ blocks, applies a **Discrete Cosine Transform (DCT)**, and deletes the high-frequency color variations to save bytes.

When stitched back together, this creates microscopic "Block Artifacts"—sharp, artificial grid lines running across the entire image every 8 pixels. 

A human eye ignores these tiny grid lines. But a CNN's first layer is literally a sequence of edge detectors (like the ones we built in Notebook 01). The CNN latches onto these intense artificial vertical and horizontal lines, completely drowning out the actual physical edges of the objects you want to detect. The feature maps shatter.

In [ ]:
import albumentations as A
import cv2
from matplotlib import pyplot as plt

# -----------------------------------------------------
# IMPLEMENTATION: The Brutal Edge Pipeline
# -----------------------------------------------------

"""
Instead of basic 'Random Cropping', a true production pipeline must simulate hardware physics.
"""

production_transform = A.Compose([
    # 1. Simulate the camera physically vibrating on a mechanical arm
    A.MotionBlur(blur_limit=15, p=0.3),
    
    # 2. Simulate ISO sensor noise during night-time shifts
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    
    # 3. Simulate sunlight glare hitting the lens at different times of day
    A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4, p=0.5),
    
    # 4. SIMULATE THE MJPEG ASSASSIN
    A.ImageCompression(quality_lower=30, quality_upper=80, p=0.4),
    
    # 5. Simulate the camera being knocked slightly out of alignment
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5)
])

print("Production hardware physics simulation pipeline defined.")
# If your model is trained on this, it will survive deployment.

## 2. MixUp & CutMix Regularization

Even with hardware simulation, models get lazy. If you want a model to classify "Dogs", and every dog in the dataset is sitting on green grass, the CNN will just learn that `Green Pixels -> Dog`. 

**MixUp:** You take an image of a Dog, and an image of a Car. You mathematically blend them together (e.g., $60\%$ opacity Dog, $40\%$ opacity Car). The target label becomes exactly `[0.6 Dog, 0.4 Car]`. This forces the network to stop relying on binary "all or nothing" spatial features and learn continuous probability boundaries.

**CutMix:** You physically cut a square box out of the Car image and paste it directly on top of the Dog's face. The label is proportional to the area. 

**Why CutMix is genius:** If a CNN is trying to detect a Dog, it usually just learns to look for the Dog's eyes and ears. By pasting a car over the dog's face, the CNN is forced to look at the dog's paws, tail, and fur to make the prediction. It forces the network to learn holistic, full-body feature representations, acting as the ultimate spatial regularization.